[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/surfaces/motor_loop/motor_loop.ipynb)

# Loop Subdivision of Motors

This example computes Loop subdivision of a triangle mesh whose vertices carry rigid placements of frames instead of points. A Loop step refines a mesh by averaging: every vertex of the finer mesh is a fixed average of vertices of the coarser one, with shares that depend only on how the mesh is connected. A ribbon wound up by a twist shows what averaging placements buys over averaging positions. Refined as points it wrings itself thin, because the average of two positions cuts across the turn between them; refined as placements it keeps its width, because the average of two frames is the frame turned halfway, out on the arc between them.

Each placement is a motor, and one Loop step is a sparse extensor with scalar cells, a map from the coarser mesh's vertex sites to the finer one's that applies to whatever sits there, `step * mesh.vertices`. A weighted sum of motors is no motor, so every averaged motor is normalized back after each step, `(step * mesh.vertices).normalized()`: average in the algebra, return to the group. The surface is the origin each motor carries, `motor >> origin`.

In the matrix notation of subdivision surfaces a Loop step reads as a sparse subdivision matrix $S$ acting on the stacked vertex positions, $P' = S P$; refining motors, the same $S$ acts on the stacked dual quaternions, each row followed by normalization.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
from collections.abc import Iterator
from functools import reduce

import numpy as np

from numga import NumpyContext, stack
from numga.algebras import PGA3D as ga
from numga.sparse import SparseExtensor
from examples import instantiate
from examples.util.animation import show_animation
from examples.surfaces.motor_loop import render

mv = NumpyContext(ga).multivector
Plane = ga.gatype.vector()
Line = ga.gatype.bivector()
Point = ga.gatype.antivector()
Motor = ga.gatype.rotor()
Mesh = instantiate("examples.util.mesh", ga).Mesh


# The ribbon's control mesh: its vertices along and across, its length and width, and its four corners,
# pinned.
n_along, n_across = 5, 3
length, width = 4.0, 1.2
corners = [0, n_across - 1, n_along * n_across - n_across, n_along * n_across - 1]

# How far along and across the ribbon each control vertex lies, row by row, and its faces: two triangles
# per cell of the grid, all turning the same way.
along, across = (grid.reshape(-1) for grid in np.meshgrid(np.linspace(-length / 2, length / 2, n_along),
                                                           np.linspace(-width / 2, width / 2, n_across), indexing="ij"))
index = np.arange(n_along * n_across).reshape(n_along, n_across)
cell, right, up, opposite = index[:-1, :-1], index[:-1, 1:], index[1:, :-1], index[1:, 1:]
control_faces = np.concatenate([np.stack([cell, right, opposite], axis=-1).reshape(-1, 3),
                                np.stack([cell, opposite, up], axis=-1).reshape(-1, 3)])
pinned = np.isin(np.arange(n_along * n_across), corners)

## 1. One map, any contents

A Loop step is fixed by the faces alone: a sparse map from the vertex sites of the coarser mesh to those of the finer one, its cells scalars, here with the pinned corners keeping themselves. The map does not care what sits at the sites. Applied to motors it averages them, and every averaged motor is normalized back to a motor after each step. Applied to the points the motors carry, nothing happens between the steps, so the steps compose into one map. The two surfaces are drawn side by side, the points on the left and the motors on the right; a frame is drawn as its origin and the ends of its three axes.

In [ ]:
# The Loop steps taken, and the length of the drawn frame axes.
n_loop_steps = 3
axis_length = 0.3

In [ ]:
def turn(plane: Line, angle: np.ndarray) -> Motor:
    """The turn by an angle in a plane through the origin."""
    return (plane * (-angle / 2)).exp()


def slide(direction: Plane, distance: np.ndarray) -> Motor:
    """The translation by a distance along the normal of a plane through the origin."""
    return (direction * mv.w * (distance / 2)).exp()


def ribbon(along: np.ndarray, across: np.ndarray, angle: np.ndarray) -> Motor:
    """Frames on a ribbon down the z axis: slid along it and out across it along x, then turned about the
    axis by an angle."""
    return turn(mv.xy, angle) * slide(mv.z, along) * slide(mv.x, across)


def refinements(mesh: Mesh, pinned: np.ndarray, n_loop_steps: int) -> Iterator[tuple[SparseExtensor, Mesh]]:
    """Each of so many Loop steps on a mesh of motors, with the vertices pinned staying where they are: the
    averaging map, built once from the faces, and the finer mesh it gives, every averaged motor normalized
    back to a motor."""
    for _ in range(n_loop_steps):
        step = mesh.loop(pinned)                                               # [finer, coarser] Scalar
        mesh, pinned = Mesh((step * mesh.vertices).normalized(), mesh.split_faces), mesh.split_pins(pinned)
        yield step, mesh


markers = stack([mv.w.dual(), mv.point([axis_length, 0.0, 0.0]), mv.point([0.0, axis_length, 0.0]),
                 mv.point([0.0, 0.0, axis_length])])                           # [axes + 1] Point
origin = markers[0]                                                            # [] Point


def refined(motors: Motor) -> tuple[Point, Point, np.ndarray]:
    """The control frames of motors at the vertices of the ribbon, and the surface refined two ways: as the
    points the motors carry, through the steps composed into one map, and as the motors themselves,
    normalized after every step; with the faces of the surfaces."""
    steps, meshes = zip(*refinements(Mesh(motors, control_faces), pinned, n_loop_steps))
    points = reduce(lambda total, step: step * total, steps) * (motors >> origin)   # [frames] Point[refined]
    surfaces = stack([points, meshes[-1].vertices >> origin], axis=1)          # [frames, schemes] Point[refined]
    return motors[:, None] >> markers, surfaces, meshes[-1].faces              # [frames, axes + 1] Point[vertices]

## 2. Winding up the ribbon

One end of the ribbon is held and the other wound one and a half turns and back, the twist building towards the wound end as the square of the distance. The points, on the left, are averaged as positions: two frames far apart in turn have their average close to the axis, and the ribbon wrings itself into a strand. The motors, on the right, are averaged as placements, and every average lands turned partway between its neighbours, at their distance from the axis. The bold markers are the control frames, x red, y green and z blue, over the dashed control net.

In dual-quaternion notation a motor is a unit dual quaternion and each step is dual-quaternion linear blending followed by normalization; refining the points is linear blend skinning, and the strand is its candy-wrapper artifact.

In [ ]:
# The turn of the far end at the top of the wind-up, the near end held; the frames of the wind-up and
# release, and the duration of each.
wound_turn = 3 * np.pi
n_frames = 48
duration_ms = 60

In [ ]:
wound = wound_turn * (1 - np.cos(np.linspace(0, 2 * np.pi, n_frames, endpoint=False))) / 2   # [frames]
held = ((along + length / 2) / length) ** 2                                    # [vertices]
winding = ribbon(along, across, wound[:, None] * held).field()                 # [frames] Motor[vertices]
control, surfaces, faces = refined(winding)                                    # [frames, schemes] Point[refined]

show_animation(render.animate(control, surfaces, control_faces, faces), "motor_loop_ribbon", duration_ms)

## 3. Turning the frames in place

Hold the ribbon flat and turn every frame back and forth about an axis of its own through its origin, each with its own phase. No control point moves, so neither does the surface of the points. The surface of the motors does: the average of two frames moves along the screw motion between them, and where neighbouring frames turn against each other that motion bends, bowing the surface out of the chord. A turn shared by all frames changes nothing, since it carries every origin to itself; only the turns between neighbours bend the surface, so the frames are no tangents the surface must follow, but a bending between neighbours.

In [ ]:
# How far each frame turns either way about its own axis as the ribbon lies flat, and the seed of those
# axes and of the turns' phases.
turning_angle = np.pi / 3
turning_seed = 2

In [ ]:
rng = np.random.default_rng(turning_seed)
axes = mv(Line.derive.nondegenerate, rng.normal(size=(len(along), 3))).normalized()   # [vertices] Line
phases = rng.uniform(0, 2 * np.pi, len(along))
angles = turning_angle * np.sin(np.linspace(0, 2 * np.pi, n_frames, endpoint=False)[:, None] + phases)   # [frames, vertices]
turning = (ribbon(along, across, 0.0) * turn(axes, angles)).field()           # [frames] Motor[vertices]
turning_control, turning_surfaces, turning_faces = refined(turning)            # [frames, schemes] Point[refined]

show_animation(render.animate(turning_control, turning_surfaces, control_faces, turning_faces), "motor_loop_turning", duration_ms)

In [ ]:
# checks
(points_rest, motors_rest), (points_peak, motors_peak) = render.reach(surfaces[[0, n_frames // 2]])
# Untwisted the two agree; wound up, the points wring the ribbon towards its axis and the motors do not.
assert abs(points_rest - motors_rest) < 1e-12
assert points_peak < 0.7 * points_rest and motors_peak >= motors_rest
# Turning the frames in place leaves the points alone and bends the motors.
turning_points, turning_motors = turning_surfaces[:, 0], turning_surfaces[:, 1]   # [frames] Point[refined]
assert render.separation(turning_points, turning_points[:1]) < 1e-12
assert render.separation(turning_motors, turning_points) > 0.05
# A turn common to every frame changes nothing.
flat = ribbon(along, across, 0.0)                                              # [vertices] Motor
common = refined(stack([flat, flat * turn(mv.yz, 0.7)]).field())[1][:, 1]     # [cases] Point[refined]
assert render.separation(common[0], common[1]) < 1e-12